# **K-Means Clustering — Seller Segmentation**

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import PowerTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import plotly.express as px
import plotly.io as pio


# =========================
# Notebook display setting
# =========================
pio.renderers.default = "notebook_connected"


# =========================
# Directory configuration
# =========================
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

NOTEBOOK_OUTPUT_DIR = PROJECT_DIR / "notebooks" / "outputs"

NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# Input file
# =========================
INPUT_FILE = NOTEBOOK_OUTPUT_DIR / "seller_scoring_bgnbd_gammagamma_production.csv"

# =========================
# Load data
# =========================
df = pd.read_csv(INPUT_FILE)

print(f"Loaded data from: {INPUT_FILE}")
print(f"Shape: {df.shape}")
display(df.head())

Loaded data from: c:\Assigment\2025-2026\Marketing-Driven\final_project\notebooks\outputs\seller_scoring_bgnbd_gammagamma_production.csv
Shape: (2248, 25)


,seller_id,frequency,recency,T,monetary_value,total_orders_cal,total_gmv_cal,historical_aov_cal,prob_alive,expected_avg_gmv,...,expected_orders_60d,activity_score_60d,p_purchase_approx_60d,predicted_gmv_60d,expected_commission_60d,expected_orders_90d,activity_score_90d,p_purchase_approx_90d,predicted_gmv_90d,expected_commission_90d
0,0015a82c2db000af6aaaf3ae2ecb0532,2,21.416308,216.071458,895.000000,3,2685.00,895.000000,0.099162,894.276630,...,0.065367,0.065367,0.063276,58.455924,8.768389,0.097420,0.097420,0.092825,87.120216,13.068032
1,001cca7ae9ae17fb1caed9dfb1094831,191,447.554861,450.204109,125.739424,192,24116.13,125.604844,0.997701,125.777572,...,25.038213,25.038213,1.000000,3149.245635,472.386845,37.397244,37.397244,1.000000,4703.734589,705.560188
2,002100f778ceb8431b7a1020ff7ab48f,49,210.498519,228.957963,24.626531,50,1216.60,24.332000,0.853171,24.798646,...,10.712664,10.712664,0.999978,265.659572,39.848936,15.948371,15.948371,1.000000,395.498008,59.324701
3,003554e2dce176b5555353e4f3555ac8,0,0.000000,136.713588,0.000000,1,120.00,120.000000,1.000000,120.000000,...,0.187662,0.187662,0.171106,22.519494,3.377924,0.279443,0.279443,0.243795,33.533129,5.029969
4,004c9cd9d87a3c30c522c48c4fc07416,153,444.125498,458.559317,124.838170,154,19220.23,124.806688,0.891200,124.885858,...,17.604879,17.604879,1.000000,2198.600440,329.790066,26.296725,26.296725,1.000000,3284.089090,492.613363


In [7]:
# =========================
# K-Means configuration
# =========================
HORIZON = 60
K = 3

FEATURE_GROUP_NAME = f"orders_purchase_commission_{HORIZON}d"

FEATURES = [
    f"expected_orders_{HORIZON}d",
    f"p_purchase_approx_{HORIZON}d",
    f"expected_commission_{HORIZON}d",
]

print("Feature group:", FEATURE_GROUP_NAME)
print("Features:", FEATURES)


# =========================
# Validate selected features
# =========================
missing_cols = [col for col in FEATURES if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")


# =========================
# Prepare feature matrix
# =========================
X_raw = df[FEATURES].copy()

X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
X_raw = X_raw.fillna(0)

# =========================
# Transform skewness + standardize
# =========================
transformer = PowerTransformer(
    method="yeo-johnson",
    standardize=True
)

X_scaled = transformer.fit_transform(X_raw)


# =========================
# Fit K-Means
# =========================
kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=20,
    max_iter=300
)

cluster_labels = kmeans.fit_predict(X_scaled)


# =========================
# Attach cluster labels
# =========================
df_kmeans = df.copy()

df_kmeans["kmeans_feature_group"] = FEATURE_GROUP_NAME
df_kmeans["kmeans_horizon"] = HORIZON
df_kmeans["kmeans_k"] = K
df_kmeans["seller_cluster"] = cluster_labels


# =========================
# Evaluation metrics
# =========================
silhouette = silhouette_score(X_scaled, cluster_labels)
davies_bouldin = davies_bouldin_score(X_scaled, cluster_labels)
calinski_harabasz = calinski_harabasz_score(X_scaled, cluster_labels)

print(f"K-Means Feature Group: {FEATURE_GROUP_NAME}")
print(f"Horizon: {HORIZON} days")
print(f"K: {K}")
print(f"Silhouette Score: {silhouette:.4f}")
print(f"Davies-Bouldin Score: {davies_bouldin:.4f}")
print(f"Calinski-Harabasz Score: {calinski_harabasz:.4f}")

Feature group: orders_purchase_commission_60d
Features: ['expected_orders_60d', 'p_purchase_approx_60d', 'expected_commission_60d']
K-Means Feature Group: orders_purchase_commission_60d
Horizon: 60 days
K: 3
Silhouette Score: 0.5295
Davies-Bouldin Score: 0.6834
Calinski-Harabasz Score: 6881.4411


In [8]:
# =========================
# Cluster profile
# =========================
cluster_profile = (
    df_kmeans
    .groupby("seller_cluster")
    .agg(
        n_sellers=("seller_cluster", "count"),

        expected_orders_60d_mean=(f"expected_orders_{HORIZON}d", "mean"),
        expected_orders_60d_median=(f"expected_orders_{HORIZON}d", "median"),

        p_purchase_60d_mean=(f"p_purchase_approx_{HORIZON}d", "mean"),
        p_purchase_60d_median=(f"p_purchase_approx_{HORIZON}d", "median"),

        expected_commission_60d_mean=(f"expected_commission_{HORIZON}d", "mean"),
        expected_commission_60d_median=(f"expected_commission_{HORIZON}d", "median"),
        expected_commission_60d_sum=(f"expected_commission_{HORIZON}d", "sum"),
    )
    .reset_index()
)

cluster_profile["seller_share"] = (
    cluster_profile["n_sellers"] / cluster_profile["n_sellers"].sum()
)

cluster_profile["expected_commission_60d_share"] = (
    cluster_profile["expected_commission_60d_sum"]
    / cluster_profile["expected_commission_60d_sum"].sum()
)

cluster_profile = cluster_profile.sort_values(
    "expected_commission_60d_median",
    ascending=False
)

display(cluster_profile)

,seller_cluster,n_sellers,expected_orders_60d_mean,expected_orders_60d_median,p_purchase_60d_mean,p_purchase_60d_median,expected_commission_60d_mean,expected_commission_60d_median,expected_commission_60d_sum,seller_share,expected_commission_60d_share
2,2,857,13.877609,6.814062,0.985546,0.998902,283.500425,135.969629,242959.863867,0.381228,0.911484
1,1,552,1.414048,1.265127,0.700281,0.717793,37.214350,23.451205,20542.321191,0.245552,0.077066
0,0,839,0.158710,0.101584,0.136266,0.096595,3.637593,1.536463,3051.940941,0.373221,0.011450


## **Clustering Inference Insights**

**Overall Distribution Summary**

> The K-Means clustering model successfully identified three distinct seller segments with significantly different behavioral and commercial characteristics. By combining: Expected future orders, Estimated purchase probability, Projected commission value within the next 60 days, the segmentation reveals strong concentration patterns in seller value contribution across the platform.

| Cluster | Segment Label | Number of Sellers | Seller Share | Total Commission (60d) | Commission Share |
|---|---|---|---|---|---|
| 2 | High-Value Active Sellers | 857 | 38.1% | ~243,000 | 91.1% |
| 1 | Mid-Tier Engaged Sellers | 552 | 24.6% | ~20,500 | 7.7% |
| 0 | Dormant Sellers | 839 | 37.3% | ~3,050 | 1.1% |

The commission distribution is extremely concentrated: only 38% of sellers (Cluster 2) generate 91% of the projected commission. This is a classic manifestation of the Pareto Principle within a marketplace ecosystem, although at a much more extreme level than the conventional 80/20 pattern — here, the ratio is approximately 91/38.

Such a high concentration level creates significant systemic business risk. If churn rates increase within Cluster 2, the platform’s future revenue performance could be heavily impacted.

---

### **Cluster 2 — High-Value Active Sellers**

#### **_Quantitative Definition_**

| Metric | Mean | Median |
|---|---|---|
| Expected orders (60d) | 13.88 orders | 6.81 orders |
| P(purchases) | 98.6% | 99.9% |
| Expected commission (60d) | 283.5 | 135.97 |

Cluster 2 represents the platform’s most commercially valuable seller segment. Sellers in this group demonstrate:
- extremely high purchase probability,
- strong expected transaction volume,
- and dominant future commission contribution.

Despite representing only a minority of the seller base, this cluster generates almost the entire projected commission of the platform. This indicates a highly concentrated revenue structure where platform performance is heavily dependent on retaining and supporting these sellers.

#### **_Behavioral Analysis_**

Cluster 2 consists of sellers with near-perfect active probability while maintaining the highest transaction frequency across the entire seller population. According to the BG/NBD model, these sellers are considered as "fully" active, meaning their transaction history shows virtually no indication of churn behavior.

The substantial gap between the mean (13.88) and median (6.81) of expected orders suggests the presence of long-tail outliers within the cluster. A small subset of top-tier sellers likely generates extremely high projected order volumes (potentially 50–200 orders), significantly inflating the mean. A similar pattern appears in expected commission confirms a strongly right-skewed distribution even within the cluster itself.

#### **_Business Implications_**

- Cluster 2 represents the platform’s core revenue engine. Any deterioration within this group — even affecting only 5–10% of sellers — could create substantial revenue impact.
- However, not all sellers inside Cluster 2 are homogeneous. Additional internal sub-segmentation should be considered to separate top outlier sellers from the remaining population, enabling more personalized account management strategies.

---

### **Cluster 1 — Mid-Tier Engaged Sellers**

#### **_Quantitative Definition_**

| Metric | Mean | Median |
|---|---|---|
| Expected orders (60d) | 1.41 orders | 1.27 orders |
| P(purchases) | 70.0% | 71.8% |
| Expected commission (60d) | 37.2 | 23.5 |

#### **_Behavioral Analysis_**

Cluster 1 is arguably the most analytically interesting segment due to its transitional characteristics.

_Observation 1 — Moderate P(purchases) (~70%)_

The BG/NBD model estimates that approximately 70% of sellers in this group are still active, implying a churn probability of roughly 30%. This segment is neither fully active nor fully churned. Instead, it occupies a transition zone:
- sellers may recover and move upward into Cluster 2 with appropriate intervention,
- or gradually decline into Cluster 0 if neglected.

_Observation 2 — Small Mean/Median Gap_

The mean-to-median ratio is relatively small. This is significantly lower than Cluster 2, indicating that seller behavior within Cluster 1 is relatively homogeneous, without extreme outliers. The average transaction frequency (~1.3 orders per 60 days) remains low. However, since P(purchases) is still around 72%, these sellers are clearly still interacting with the platform.

The issue may therefore lie in conversion efficiency rather than seller inactivity itself.

#### **_Business Implications_**

Cluster 1 represents the platform’s most promising growth opportunity with potentially strong ROI on intervention efforts.

The cost of activation is likely much lower than acquiring entirely new sellers, while the probability of upgrading these sellers into Cluster 2 remains realistic if operational friction points are addressed.

A key business question emerges: Why is P(purchases) relatively high while order volume remains low?

Several hypotheses should be investigated:

- Sellers experience friction during listing or onboarding.
- Sellers operate seasonally and current data reflects off-season behavior.
- Sellers receive insufficient traffic or marketplace visibility.
- Sellers prioritize competing platforms over the current marketplace.

---

### **Cluster 0 — Dormant Sellers**

#### **_Quantitative Definition_**

| Metric | Mean | Median |
|---|---|---|
| Expected orders (60d) | 0.16 orders | 0.10 orders |
| P(purchases) | 13.6% | 9.7% |
| Expected commission (60d) | 3.64 | 1.54 |

#### **_Behavioral Analysis_**

Cluster 0 contains sellers that the BG/NBD model classifies as having extremely high churn probability.

A median P(purchases) of only 9.7% suggests that, probabilistically, over 90% of sellers in this cluster are effectively inactive. This means that the elapsed time since their last transaction is disproportionately long relative to their historical purchasing behavior.

An expected order value of only 0.10 orders over 60 days effectively indicates that the model predicts almost no future transactions from these sellers. Even the mean value (0.16) remains extremely low, confirming the absence of meaningful outliers within the cluster. Importantly, 37.3% of all sellers belong to this cluster, yet they contribute only 1.1% of projected commission.

In practical terms, if the entire Cluster 0 population churned immediately, the revenue impact (~3,050 commission) would roughly equal the contribution of a single average seller from Cluster 2.

#### **_Business Implications_**

The primary risk associated with Cluster 0 is not revenue loss, but operational inefficiency and opportunity cost. Resources allocated toward these sellers — customer support, marketing exposure, infrastructure, and operational maintenance — may produce substantially higher returns if redirected toward Clusters 1 or 2.

However, immediate offboarding should be avoided until additional analysis distinguishes between two important subgroups:

- Seasonal Sellers: Some sellers may only operate during seasonal periods (e.g., holidays, back-to-school periods). In such cases, low P(purchase) may simply reflect temporary inactivity during off-season periods.
- Truly Churned Sellers: Other sellers demonstrate no seasonal patterns and exhibit prolonged inactivity without explanation. These sellers are more appropriate candidates for offboarding or minimal-maintenance win-back strategies.

## **Final Conclusion**

The K-Means segmentation framework successfully identified meaningful seller groups with substantial differences in:

- future transaction behavior,
- purchase probability,
- and expected commission contribution.

This segmentation enables the platform to implement more efficient and data-driven seller management strategies, improve retention effectiveness, and optimize long-term commission growth.

In [12]:
import pandas as pd
import plotly.express as px
from IPython.display import HTML, display


# =========================
# Configuration
# =========================
HORIZON = 60

FEATURE_GROUP_NAME = f"orders_purchase_commission_{HORIZON}d"

FEATURES = [
    f"expected_orders_{HORIZON}d",
    f"p_purchase_approx_{HORIZON}d",
    f"expected_commission_{HORIZON}d",
]


# =========================
# Validate required columns
# =========================
required_cols = FEATURES + ["seller_cluster"]

missing_cols = [col for col in required_cols if col not in df_kmeans.columns]

if missing_cols:
    raise ValueError(f"Missing columns in df_kmeans: {missing_cols}")


# =========================
# Infer K dynamically
# =========================
K = df_kmeans["seller_cluster"].nunique()

# =========================
# Prepare RAW plot dataframe
# =========================
plot_raw_df = df_kmeans.copy()
plot_raw_df["seller_cluster"] = plot_raw_df["seller_cluster"].astype(str)


# =========================
# Hover columns
# =========================
hover_cols = FEATURES.copy()

if "seller_id" in plot_raw_df.columns:
    hover_cols = ["seller_id"] + hover_cols


# =========================
# Plot 1: RAW business space
# Better axis arrangement
# =========================
fig_raw = px.scatter_3d(
    plot_raw_df,
    x="expected_commission_60d",
    y="expected_orders_60d",
    z="p_purchase_approx_60d",
    color="seller_cluster",
    opacity=0.65,
    title=f"3D K-Means Clusters - Raw Business Space | {FEATURE_GROUP_NAME}, K={K}",
    hover_data=hover_cols,
)

fig_raw.update_traces(
    marker=dict(size=3)
)

fig_raw.update_layout(
    width=1100,
    height=800,
    legend_title="Seller Cluster",
    scene=dict(
        xaxis_title="Expected Commission 60d",
        yaxis_title="Expected Orders 60d",
        zaxis_title="P(Purchase) Approx 60d",
        aspectmode="manual",
        aspectratio=dict(x=1.4, y=1.2, z=0.9),
        camera=dict(
            eye=dict(x=1.8, y=1.6, z=1.1)
        )
    ),
    margin=dict(l=0, r=0, b=0, t=60)
)


# =========================
# Validate X_scaled exists
# =========================
if "X_scaled" not in globals():
    raise ValueError(
        "X_scaled not found. Please run the K-Means training cell first."
    )


# =========================
# Prepare SCALED plot dataframe
# Reuse exact X_scaled from training
# =========================
scaled_feature_names = [
    "scaled_expected_orders_60d",
    "scaled_p_purchase_approx_60d",
    "scaled_expected_commission_60d",
]

plot_scaled_df = pd.DataFrame(
    X_scaled,
    columns=scaled_feature_names,
    index=df_kmeans.index
)

plot_scaled_df["seller_cluster"] = (
    df_kmeans["seller_cluster"].astype(str)
)


# =========================
# Add raw values for hover
# =========================
for col in FEATURES:
    plot_scaled_df[col] = df_kmeans[col].values

if "seller_id" in df_kmeans.columns:
    plot_scaled_df["seller_id"] = df_kmeans["seller_id"].values


# =========================
# Hover columns for scaled plot
# =========================
hover_cols_scaled = FEATURES.copy()

if "seller_id" in plot_scaled_df.columns:
    hover_cols_scaled = ["seller_id"] + hover_cols_scaled


# =========================
# Plot 2: Scaled K-Means model space
# =========================
fig_scaled = px.scatter_3d(
    plot_scaled_df,
    x="scaled_expected_commission_60d",
    y="scaled_expected_orders_60d",
    z="scaled_p_purchase_approx_60d",
    color="seller_cluster",
    opacity=0.65,
    title=f"3D K-Means Clusters - Scaled Model Space | {FEATURE_GROUP_NAME}, K={K}",
    hover_data=hover_cols_scaled,
)

fig_scaled.update_traces(
    marker=dict(size=3)
)

fig_scaled.update_layout(
    width=1100,
    height=800,
    legend_title="Seller Cluster",
    scene=dict(
        xaxis_title="Scaled Expected Commission 60d",
        yaxis_title="Scaled Expected Orders 60d",
        zaxis_title="Scaled P(Purchase) Approx 60d",
        aspectmode="manual",
        aspectratio=dict(x=1.4, y=1.2, z=0.9),
        camera=dict(
            eye=dict(x=1.8, y=1.6, z=1.1)
        )
    ),
    margin=dict(l=0, r=0, b=0, t=60)
)


# =========================
# Display inline in notebook
# =========================
display(HTML("<h3>Raw Business Feature Space</h3>"))
display(HTML(fig_raw.to_html(
    full_html=False,
    include_plotlyjs="cdn"
)))

## **Analysis of the 3D Clustering Visualization**

### **Raw Business Feature Space**

The 3D visualization in the raw business feature space reveals a highly imbalanced data structure. Most observations are vertically compressed along the purchase probability axis, while expected orders and expected commission values exhibit extremely wide horizontal dispersion.

This pattern indicates severe right-skewness in the original feature distributions. A very small number of sellers generate exceptionally large commission values — in some cases reaching several thousand units — while the majority of sellers remain concentrated near zero.

As a consequence, the Euclidean distance calculation used by the K-Means algorithm becomes heavily dominated by the commission dimension. In practical terms, the clustering process in the raw feature space is disproportionately influenced by seller commission magnitude rather than balanced behavioral similarity across all features.

Within this representation:

- **Cluster 2 (red)** appears as a dispersed group of high-commission outliers, consistent with the behavioral profile of strategically valuable “top-tier” sellers.
- **Cluster 0 (blue)** and **Cluster 1 (green)** are compressed into a narrow region near the origin, making their separation visually ambiguous in the unscaled space.

This observation highlights one of the major limitations of directly applying distance-based clustering algorithms to highly skewed marketplace data without prior transformation.

In [ ]:
display(HTML("<h3>Scaled K-Means Model Space</h3>"))
display(HTML(fig_scaled.to_html(
    full_html=False,
    include_plotlyjs=False
)))

### **Scaled Model Space — Effective Behavioral Separation After Transformation**

After applying the Yeo-Johnson transformation followed by standardization, the clustering structure becomes substantially clearer. In the transformed feature space, the three seller segments separate along a diagonal gradient extending from the lower-left region (low expected orders, low purchase probability, and low commission) toward the upper-right region (high values across all dimensions).

This pattern is behaviorally meaningful and indicates that the three predictive variables are positively correlated. More importantly, it demonstrates that the transformation process successfully reduced scale imbalance and enabled the K-Means algorithm to learn underlying seller behavior rather than merely detecting extreme numerical outliers.

Compared to the raw feature space, the scaled representation produces substantially more stable and interpretable clusters.

However, significant overlap is still observed between **Cluster 2 (red)** and **Cluster 1 (green)** in the middle region. The boundary is not sharp, meaning some sellers in the intermediate region may be assigned to the wrong cluster depending on the run.